AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors.

# Linear regression: train / validation / test

Trains OLS and Lasso baselines, tunes Lasso's alpha by scoring candidates against the held-out
**validation** set,retrains at the best alpha, and scores the tuned model plus a persistence baseline against the
**test** set.

Set `HORIZON` below to `'0h'`, `'3h'`, or `'6h'` and re-run the notebook top-to-bottom for each
horizon. The results CSV accumulates across horizons (each run replaces only that horizon's rows);
the saved models, feature-importance CSV, and bootstrap-prediction export are horizon-suffixed files
that don't collide across runs.

# I. Imports

In [17]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# II. Configuration

In [18]:
platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

MODELS_DIR = Path('work/models')

In [19]:
HORIZON_CONFIG = {
    '0h': {
        'train_path': PROCESSED_DIR / 'lr_features_0h_train.parquet',
        'validation_path': PROCESSED_DIR / 'lr_features_0h_validation.parquet',
        'test_path': PROCESSED_DIR / 'lr_features_0h_test.parquet',
        'target': 'kp_index',
        'cols_to_drop': ['kp_10', 'year', 'day', 'hour', 'minute', 'minute_cos', 'minute_sin', 'data_split'],
        'model_suffix': '0hr',
        'persistence_horizon_hours': 0,
    },
    '3h': {
        'train_path': PROCESSED_DIR / 'lr_features_3h_train.parquet',
        'validation_path': PROCESSED_DIR / 'lr_features_3h_validation.parquet',
        'test_path': PROCESSED_DIR / 'lr_features_3h_test.parquet',
        'target': 'kp_index_3_hr_forecast',
        'cols_to_drop': ['kp_10', 'kp_index', 'year', 'day', 'hour', 'minute', 'minute_cos', 'minute_sin', 'data_split'],
        'model_suffix': '3hr',
        'persistence_horizon_hours': 3,
    },
    '6h': {
        'train_path': PROCESSED_DIR / 'lr_features_6h_train.parquet',
        'validation_path': PROCESSED_DIR / 'lr_features_6h_validation.parquet',
        'test_path': PROCESSED_DIR / 'lr_features_6h_test.parquet',
        'target': 'kp_index_6_hr_forecast',
        'cols_to_drop': ['kp_10', 'kp_index', 'year', 'day', 'hour', 'minute', 'minute_cos', 'minute_sin', 'data_split'],
        'model_suffix': '6hr',
        'persistence_horizon_hours': 6,
    },
}

In [20]:
# Set the horizon to run (change and re-run the notebook top-to-bottom for each horizon)
HORIZON = '0h'
config = HORIZON_CONFIG[HORIZON]

ALPHA_GRID = np.logspace(-4, 1, 50)

RESULTS_CSV_PATH = PROCESSED_DIR / 'lr_train_val_test_results.csv'
OLS_MODEL_PATH = MODELS_DIR / f'best_ols_model_{config["model_suffix"]}.pkl'
LASSO_MODEL_PATH = MODELS_DIR / f'best_lasso_model_{config["model_suffix"]}.pkl'
FEATURE_IMPORTANCE_PATH = PROCESSED_DIR / f'lasso_feature_importance_{HORIZON}.csv'
BOOTSTRAP_EXPORT_PATH = PROCESSED_DIR / f'bootstrap_predictions_{HORIZON}.csv'

print(f'Horizon: {HORIZON}')
print(f'Target column: {config["target"]}')
print(f'Columns dropped from features: {config["cols_to_drop"]}')
print(f'Results CSV (accumulated): {RESULTS_CSV_PATH}')
print(f'OLS model path: {OLS_MODEL_PATH}')
print(f'Lasso model path: {LASSO_MODEL_PATH} (overwrites the CV-tuned model from 07a, if present)')
print(f'Feature importance path: {FEATURE_IMPORTANCE_PATH}')
print(f'Bootstrap export path: {BOOTSTRAP_EXPORT_PATH}')

Horizon: 0h
Target column: kp_index
Columns dropped from features: ['kp_10', 'year', 'day', 'hour', 'minute', 'minute_cos', 'minute_sin', 'data_split']
Results CSV (accumulated): work\Processed\lr_train_val_test_results.csv
OLS model path: work\models\best_ols_model_0hr.pkl
Lasso model path: work\models\best_lasso_model_0hr.pkl (overwrites the CV-tuned model from 07a, if present)
Feature importance path: work\Processed\lasso_feature_importance_0h.csv
Bootstrap export path: work\Processed\bootstrap_predictions_0h.csv


# III. Function definitions

## 1. `load_split`

In [21]:
def load_split(path, dataset_name):
    """Load one lr_features parquet split, sorted by its datetime index."""

    df = pd.read_parquet(path).sort_index()

    print(f'--- load_split: {dataset_name} ---')
    print(f'Path: {path}')
    print(f'Shape: {df.shape}, range: {df.index.min()} to {df.index.max()}')

    return df

## 2. `prep_model_data`

In [22]:
def prep_model_data(df, dataset_name, cols_to_drop, target_var):
    """Split a loaded frame into (X, y), dropping nulls and non-feature columns."""

    rows_before = len(df)
    df = df.dropna()
    rows_after = len(df)

    df = df.drop(columns=cols_to_drop)

    y_data = df[target_var]
    X_data = df.drop(columns=[target_var])

    print(f'X/y data prepared for {dataset_name}.')
    print(f'Records before dropping nulls: {rows_before}.')
    print(f'Records after dropping nulls: {rows_after}.')
    print('Target column in y:', y_data.name)
    print('Feature columns in X (first 10):\n', X_data.columns[0:10])

    return X_data, y_data

## 3. `fit_ols_pipeline`

In [23]:
def fit_ols_pipeline(X_train, y_train):
    """Fit a StandardScaler + LinearRegression pipeline (the OLS baseline)."""

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('ols', LinearRegression()),
    ])
    pipeline.fit(X_train, y_train)

    step_names = [name for name, _ in pipeline.steps]
    print(f'Fitted OLS pipeline. Steps: {step_names}. X_train shape: {X_train.shape}')

    return pipeline

## 4. `fit_lasso_pipeline`

In [24]:
def fit_lasso_pipeline(X_train, y_train, alpha, max_iter=10000):
    """Fit a StandardScaler + Lasso pipeline at a given alpha."""

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('lasso', Lasso(alpha=alpha, max_iter=max_iter, random_state=42)),
    ])
    pipeline.fit(X_train, y_train)

    return pipeline

## 5. `evaluate_model`

In [25]:
def evaluate_model(y_true, y_pred, dataset_name):
    """Compute RMSE/MAE/R2 for a set of predictions, printing and returning both metrics and y_pred."""

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f'--- evaluate_model: {dataset_name} ---')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE:  {mae:.4f}')
    print(f'R2:   {r2:.4f}')

    metrics = {'n': len(y_true), 'rmse': rmse, 'mae': mae, 'r2': r2}
    return metrics, y_pred

## 6. `search_best_alpha`

In [26]:
def search_best_alpha(X_train, y_train, X_val, y_val, alpha_grid):
    """Manual grid search: fit Lasso on train at each alpha, score against validation, keep the best."""

    records = []
    for alpha in alpha_grid:
        pipeline = fit_lasso_pipeline(X_train, y_train, alpha=alpha)
        y_pred_val = pipeline.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
        mae = mean_absolute_error(y_val, y_pred_val)
        r2 = r2_score(y_val, y_pred_val)

        records.append({'alpha': alpha, 'rmse': rmse, 'mae': mae, 'r2': r2})
        print(f'alpha={alpha:.6f}  val_rmse={rmse:.4f}  val_mae={mae:.4f}  val_r2={r2:.4f}')

    sweep_df = pd.DataFrame(records)
    best_row = sweep_df.loc[sweep_df['rmse'].idxmin()]
    best_alpha = best_row['alpha']

    print(f'\nBest alpha (lowest validation RMSE): {best_alpha:.6f}')

    return best_alpha, sweep_df

## 7. `get_persistence_pairs`

In [27]:
def get_persistence_pairs(df, horizon_hours):
    """
    Build the (y_true, y_pred) pair for a persistence forecast at one horizon.

    y_pred is always 'the last known actual kp_index'. For horizon_hours=0 that means the previous
    row's kp_index (lag-1); for 3/6 the target columns are already forward-shifted, so the current
    row's kp_index lines up directly against kp_index_{h}_hr_forecast.
    """
    if horizon_hours == 0:
        y_true = df['kp_index']
        y_pred = df['kp_index'].shift(1)
    else:
        y_true = df[f'kp_index_{horizon_hours}_hr_forecast']
        y_pred = df['kp_index']

    mask = y_true.notna() & y_pred.notna()
    n_dropped = (~mask).sum()

    print(f'--- get_persistence_pairs: h={horizon_hours}h ---')
    print(f'Rows before pairing: {len(df):,}')
    print(f'Rows dropped (missing true/pred): {n_dropped:,}')
    print(f'Rows after pairing: {mask.sum():,}')

    return y_true[mask], y_pred[mask]

## 8. `upsert_results`

In [28]:
def upsert_results(results_df_new, csv_path, horizon):
    """Write results_df_new into csv_path, replacing any existing rows for this horizon."""

    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        existing = existing[existing['horizon'] != horizon]
        combined = pd.concat([existing, results_df_new], ignore_index=True)
    else:
        combined = results_df_new.copy()

    combined = combined.sort_values(['horizon', 'stage', 'model']).reset_index(drop=True)

    assert (combined['horizon'] == horizon).sum() == len(results_df_new), (
        'Row count for this horizon after upsert does not match the newly computed rows - '
        'check for a horizon string mismatch.'
    )

    combined.to_csv(csv_path, index=False)
    print(f'Results upserted for horizon={horizon}. Total rows in {csv_path}: {len(combined)}.')
    print(f'alpha column dtype: {combined["alpha"].dtype}')

    return combined

## 9. `export_bootstrap_predictions`

In [29]:
def export_bootstrap_predictions(pairs, path):
    """
    Export (y_true, y_pred) pairs for multiple models to one long-format CSV, for later bootstrap
    RMSE resampling (see zz_model_comparison_bootstrap.ipynb).

    pairs : dict[str, tuple(array-like, array-like)]
        Maps model name -> (y_true, y_pred). Series/arrays may differ in length across models
        (e.g. persistence drops different rows than OLS/Lasso) - long format handles this fine.
    """
    frames = []
    for model_name, (y_true, y_pred) in pairs.items():
        frames.append(pd.DataFrame({
            'model': model_name,
            'y_true': np.asarray(y_true),
            'y_pred': np.asarray(y_pred),
        }))

    export_df = pd.concat(frames, ignore_index=True)
    export_df.to_csv(path, index=False)

    print(f'Bootstrap predictions exported to {path}. Rows: {len(export_df)}.')
    print(export_df.groupby('model').size())

    return export_df

## 10. `export_feature_importance`

In [30]:
def export_feature_importance(pipeline, feature_names, path):
    """Export Lasso coefficients, sorted by absolute value descending, to CSV."""

    lasso_coefs = pipeline.named_steps['lasso'].coef_

    coef_df = pd.DataFrame({'feature': feature_names, 'coef': lasso_coefs})
    coef_df = coef_df.reindex(
        coef_df['coef'].abs().sort_values(ascending=False).index
    ).reset_index(drop=True)

    zero_count = (coef_df['coef'] == 0).sum()
    print(f'Lasso zeroed out {zero_count} of {len(coef_df)} features.')

    coef_df.to_csv(path, index=False)
    print(f'Feature importance saved to {path}')

    return coef_df

# IV. Load train, validation, and test sets

In [31]:
df_train = load_split(config['train_path'], 'train')
df_validation = load_split(config['validation_path'], 'validation')
df_test = load_split(config['test_path'], 'test')

--- load_split: train ---
Path: work\Processed\lr_features_0h_train.parquet
Shape: (55234, 85), range: 2000-01-01 06:00:00 to 2019-12-31 21:00:00
--- load_split: validation ---
Path: work\Processed\lr_features_0h_validation.parquet
Shape: (9713, 85), range: 2020-01-01 06:00:00 to 2023-06-30 21:00:00
--- load_split: test ---
Path: work\Processed\lr_features_0h_test.parquet
Shape: (8158, 85), range: 2023-07-01 18:00:00 to 2026-06-30 21:00:00


In [32]:
X_train, y_train = prep_model_data(df_train, 'train', config['cols_to_drop'], config['target'])
X_validation, y_validation = prep_model_data(df_validation, 'validation', config['cols_to_drop'], config['target'])
X_test, y_test = prep_model_data(df_test, 'test', config['cols_to_drop'], config['target'])

X/y data prepared for train.
Records before dropping nulls: 55234.
Records after dropping nulls: 55234.
Target column in y: kp_index
Feature columns in X (first 10):
 Index(['day_cos', 'day_sin', 'hour_cos', 'hour_sin', 'mag_avg_nt_log_avg_0_1h',
       'mag_avg_nt_log_min_0_1h', 'mag_avg_nt_log_max_0_1h',
       'bx_gsm_nt_avg_0_1h', 'bx_gsm_nt_min_0_1h', 'bx_gsm_nt_max_0_1h'],
      dtype='str')
X/y data prepared for validation.
Records before dropping nulls: 9713.
Records after dropping nulls: 9713.
Target column in y: kp_index
Feature columns in X (first 10):
 Index(['day_cos', 'day_sin', 'hour_cos', 'hour_sin', 'mag_avg_nt_log_avg_0_1h',
       'mag_avg_nt_log_min_0_1h', 'mag_avg_nt_log_max_0_1h',
       'bx_gsm_nt_avg_0_1h', 'bx_gsm_nt_min_0_1h', 'bx_gsm_nt_max_0_1h'],
      dtype='str')
X/y data prepared for test.
Records before dropping nulls: 8158.
Records after dropping nulls: 8158.
Target column in y: kp_index
Feature columns in X (first 10):
 Index(['day_cos', 'day_sin', 'h

# V. Baseline models (stage='baseline')

In [17]:
# OLS - fit once here; its test-set result is reused for both the 'baseline' and 'test' stage rows
ols_pipeline = fit_ols_pipeline(X_train, y_train)
ols_test_metrics, y_pred_ols_test = evaluate_model(y_test, ols_pipeline.predict(X_test), 'OLS test (baseline)')

Fitted OLS pipeline. Steps: ['scaler', 'ols']. X_train shape: (55232, 76)
--- evaluate_model: OLS test (baseline) ---
RMSE: 1.0582
MAE:  0.8021
R2:   0.4112


In [18]:
# Lasso baseline (alpha=1.0) - a throwaway comparison point, not saved or used in the bootstrap export
lasso_baseline_pipeline = fit_lasso_pipeline(X_train, y_train, alpha=1.0)
lasso_baseline_test_metrics, y_pred_lasso_baseline_test = evaluate_model(
    y_test, lasso_baseline_pipeline.predict(X_test), 'Lasso alpha=1.0 test (baseline)'
)

--- evaluate_model: Lasso alpha=1.0 test (baseline) ---
RMSE: 1.4337
MAE:  1.0887
R2:   -0.0809


# VI. Alpha search against the validation set (stage='validation')

In [19]:
best_alpha, sweep_df = search_best_alpha(X_train, y_train, X_validation, y_validation, ALPHA_GRID)
sweep_df.round(6)

alpha=0.000100  val_rmse=0.9751  val_mae=0.7569  val_r2=0.3559


alpha=0.000126  val_rmse=0.9751  val_mae=0.7569  val_r2=0.3559


alpha=0.000160  val_rmse=0.9750  val_mae=0.7568  val_r2=0.3560


alpha=0.000202  val_rmse=0.9750  val_mae=0.7568  val_r2=0.3560


alpha=0.000256  val_rmse=0.9749  val_mae=0.7567  val_r2=0.3561


...

alpha=0.596362  val_rmse=1.1725  val_mae=0.9673  val_r2=0.0687
alpha=0.754312  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=0.954095  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=1.206793  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219


alpha=1.526418  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=1.930698  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=2.442053  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=3.088844  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219


alpha=3.906940  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=4.941713  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=6.250552  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219
alpha=7.906043  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219


alpha=10.000000  val_rmse=1.2282  val_mae=1.0159  val_r2=-0.0219

Best alpha (lowest validation RMSE): 0.000829


,alpha,rmse,mae,r2
0,0.000100,0.975089,0.756893,0.355881
1,0.000126,0.975052,0.756852,0.355930
2,0.000160,0.975017,0.756810,0.355977
3,0.000202,0.974977,0.756760,0.356029
4,0.000256,0.974938,0.756704,0.356080
5,0.000324,0.974916,0.756669,0.356109
6,0.000409,0.974878,0.756646,0.356160
7,0.000518,0.974824,0.756613,0.356232
8,0.000655,0.974771,0.756578,0.356302
9,0.000829,0.974755,0.756570,0.356323


In [20]:
# Explicit retrain at the best alpha found above
lasso_tuned_pipeline = fit_lasso_pipeline(X_train, y_train, alpha=best_alpha)
lasso_tuned_val_metrics, y_pred_lasso_tuned_val = evaluate_model(
    y_validation, lasso_tuned_pipeline.predict(X_validation), 'Lasso tuned validation'
)

--- evaluate_model: Lasso tuned validation ---
RMSE: 0.9748
MAE:  0.7566
R2:   0.3563


# VII. Test-set evaluation (stage='test')

In [21]:
lasso_tuned_test_metrics, y_pred_lasso_tuned_test = evaluate_model(
    y_test, lasso_tuned_pipeline.predict(X_test), 'Lasso tuned test'
)

# OLS's test score was already computed in section V (ols_test_metrics / y_pred_ols_test) - reused
# here for the 'test' stage row rather than re-fitting or re-predicting.

--- evaluate_model: Lasso tuned test ---
RMSE: 1.0566
MAE:  0.8010
R2:   0.4130


In [22]:
# Persistence baseline - run on the raw loaded df_test (before prep_model_data dropped/split columns)
y_true_persistence, y_pred_persistence = get_persistence_pairs(df_test, config['persistence_horizon_hours'])
persistence_test_metrics, _ = evaluate_model(y_true_persistence, y_pred_persistence, 'Persistence test')

--- get_persistence_pairs: h=6h ---
Rows before pairing: 8,156
Rows dropped (missing true/pred): 0
Rows after pairing: 8,156
--- evaluate_model: Persistence test ---
RMSE: 1.1645
MAE:  0.8779
R2:   0.2869


# VIII. Results table

In [23]:
def build_results_row(horizon, stage, model, alpha, metrics):
    return {'horizon': horizon, 'stage': stage, 'model': model, 'alpha': alpha, **metrics}


rows = [
    build_results_row(HORIZON, 'baseline', 'OLS', np.nan, ols_test_metrics),
    build_results_row(HORIZON, 'baseline', 'Lasso', 1.0, lasso_baseline_test_metrics),
    build_results_row(HORIZON, 'validation', 'Lasso', best_alpha, lasso_tuned_val_metrics),
    build_results_row(HORIZON, 'test', 'OLS', np.nan, ols_test_metrics),
    build_results_row(HORIZON, 'test', 'Lasso', best_alpha, lasso_tuned_test_metrics),
    build_results_row(HORIZON, 'test', 'Persistence', np.nan, persistence_test_metrics),
]
results_df_new = pd.DataFrame(rows)[['horizon', 'stage', 'model', 'alpha', 'n', 'rmse', 'mae', 'r2']]
results_df_new

,horizon,stage,model,alpha,n,rmse,mae,r2
0,6h,baseline,OLS,NaN,8156,1.058186,0.802093,0.411198
1,6h,baseline,Lasso,1.000000,8156,1.433744,1.088690,-0.080908
2,6h,validation,Lasso,0.000829,9711,0.974755,0.756570,0.356323
3,6h,test,OLS,NaN,8156,1.058186,0.802093,0.411198
4,6h,test,Lasso,0.000829,8156,1.056583,0.801023,0.412980
5,6h,test,Persistence,NaN,8156,1.164518,0.877869,0.286920


In [24]:
results_df_all = upsert_results(results_df_new, RESULTS_CSV_PATH, HORIZON)

Results upserted for horizon=6h. Total rows in work\Processed\lr_train_val_test_results.csv: 18.
alpha column dtype: float64


,horizon,stage,model,alpha,n,rmse,mae,r2
0,0h,baseline,Lasso,1.000000,8158,1.433589,1.088578,-0.080782
1,0h,baseline,OLS,NaN,8158,0.756625,0.575275,0.698942
2,0h,test,Lasso,0.000829,8158,0.755945,0.575118,0.699483
3,0h,test,OLS,NaN,8158,0.756625,0.575275,0.698942
4,0h,test,Persistence,NaN,8157,0.907460,0.678693,0.566957
5,0h,validation,Lasso,0.000829,9713,0.686798,0.533851,0.680465
6,3h,baseline,Lasso,1.000000,8157,1.433660,1.088616,-0.080857
7,3h,baseline,OLS,NaN,8157,0.958009,0.727478,0.517369
8,3h,test,Lasso,0.002121,8157,0.956665,0.727085,0.518722
9,3h,test,OLS,NaN,8157,0.958009,0.727478,0.517369


# IX. Save models

In [25]:
joblib.dump(ols_pipeline, OLS_MODEL_PATH)
print(f'OLS model saved to {OLS_MODEL_PATH}')

joblib.dump(lasso_tuned_pipeline, LASSO_MODEL_PATH)
print(f'Lasso model saved to {LASSO_MODEL_PATH} (overwrites the prior CV-tuned model, if present)')

OLS model saved to work\models\best_ols_model_6hr.pkl
Lasso model saved to work\models\best_lasso_model_6hr.pkl (overwrites the prior CV-tuned model, if present)


# X. Feature importance

In [26]:
coef_df = export_feature_importance(lasso_tuned_pipeline, X_train.columns, FEATURE_IMPORTANCE_PATH)
coef_df.head(20)

Lasso zeroed out 27 of 76 features.
Feature importance saved to work\Processed\lasso_feature_importance_6h.csv


,feature,coef
0,flow_speed_km_s_log_max_0_1h,0.600074
1,mag_avg_nt_log_max_0_1h,0.467124
2,proton_density_n_cc_log_min_0_1h,0.272073
3,proton_density_n_cc_log_max_0_1h,0.243627
4,proton_density_n_cc_log_avg_1_2h,-0.094013
5,flow_speed_km_s_log_max_3_4h,0.091986
6,bz_gsm_nt_min_0_1h,-0.072463
7,hour_sin,-0.066995
8,mag_avg_nt_log_min_0_1h,0.065993
9,proton_density_n_cc_log_min_1_2h,-0.065006


# XI. Bootstrap prediction export

Exports (y_true, y_pred) pairs for OLS, tuned Lasso, and persistence on the test set, for later
bootstrap RMSE resampling.

In [27]:
bootstrap_pairs = {
    'OLS': (y_test.to_numpy(), y_pred_ols_test),
    'Lasso': (y_test.to_numpy(), y_pred_lasso_tuned_test),
    'Persistence': (y_true_persistence.to_numpy(), y_pred_persistence.to_numpy()),
}
export_df = export_bootstrap_predictions(bootstrap_pairs, BOOTSTRAP_EXPORT_PATH)

Bootstrap predictions exported to work\Processed\bootstrap_predictions_6h.csv. Rows: 24468.
model
Lasso          8156
OLS            8156
Persistence    8156
dtype: int64


# XII. Verification

In [28]:
print(f'Lasso baseline (alpha=1.0) test RMSE: {lasso_baseline_test_metrics["rmse"]:.4f}')
print(f'Lasso tuned (alpha={best_alpha:.6f}) test RMSE: {lasso_tuned_test_metrics["rmse"]:.4f}')

assert len(results_df_all) % 6 == 0, 'Expected 6 rows per horizon in the accumulated results CSV'
assert results_df_all['horizon'].nunique() * 6 == len(results_df_all), (
    'Row count does not match horizon count * 6 - possible duplication from a failed upsert'
)
print(f'PASS: results CSV has {len(results_df_all)} rows across {results_df_all["horizon"].nunique()} horizon(s), as expected.')

assert not export_df.isna().any().any(), 'Bootstrap export should not contain any NaNs'
print('PASS: bootstrap export has no NaNs.')

Lasso baseline (alpha=1.0) test RMSE: 1.4337
Lasso tuned (alpha=0.000829) test RMSE: 1.0566
PASS: results CSV has 18 rows across 3 horizon(s), as expected.
PASS: bootstrap export has no NaNs.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>